# InsureGig - Weekly Premium Model Training (Phase 2)
Run all cells top-to-bottom in Google Colab.

Requirements from mentors implemented in this notebook:
- Weekly premium range locked to Rs20-Rs50
- Base formula: trigger_probability * avg_income_lost_per_day * days_exposed
- Adjustments for city, peril type, worker activity tier


In [ ]:
!pip install tensorflowjs scikit-learn -q
print('Dependencies installed')


In [ ]:
# Upload exactly these CSV files from your local machine
# train.csv, Rider-Info.csv, Food_Delivery_Times.csv
from google.colab import files
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))


In [ ]:
import os
required = ['train.csv', 'Rider-Info.csv', 'Food_Delivery_Times.csv']
missing = [f for f in required if not os.path.exists(f)]
if missing:
    raise ValueError(f'Missing files: {missing}')
print('All required datasets are available.')


In [ ]:
import json
import os
from math import radians, sin, cos, sqrt, atan2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

CITY_ENC = {'Metropolitian': 1.0, 'Urban': 0.5, 'Semi-Urban': 0.0}
WEATHER_ENC = {'Sunny': 0.0, 'Cloudy': 0.17, 'Windy': 0.33, 'Fog': 0.50, 'Sandstorms': 0.67, 'Stormy': 1.0}
TRAFFIC_ENC = {'Low': 0.0, 'Medium': 0.33, 'High': 0.67, 'Jam': 1.0}
VEHICLE_ENC = {'electric_scooter': 0.0, 'bicycle': 0.33, 'scooter': 0.67, 'motorcycle': 1.0}
TOD_ENC = {'Morning': 0.0, 'Afternoon': 0.33, 'Evening': 0.67, 'Night': 1.0}

CITY_INCOME = {'Metropolitian': 700, 'Urban': 550, 'Semi-Urban': 420}
CITY_TRIGGER_BASE = {'Metropolitian': 0.20, 'Urban': 0.15, 'Semi-Urban': 0.11}
CITY_FACTOR = {'Metropolitian': 1.12, 'Urban': 1.00, 'Semi-Urban': 0.90}

PERIL_TRIGGER_FACTOR = {'weather': 1.20, 'aqi': 1.12, 'traffic': 1.08, 'platform_outage': 0.95}
PERIL_LOSS_FRACTION = {'weather': 0.24, 'aqi': 0.20, 'traffic': 0.16, 'platform_outage': 0.26}
PERIL_FACTOR = {'weather': 1.15, 'aqi': 1.08, 'traffic': 1.03, 'platform_outage': 1.00}
PERIL_EXPOSURE_DAYS = {'weather': 3, 'aqi': 2, 'traffic': 2, 'platform_outage': 1}

ACTIVITY_TRIGGER_FACTOR = {'low': 0.90, 'medium': 1.00, 'high': 1.12}
ACTIVITY_FACTOR = {'low': 0.90, 'medium': 1.00, 'high': 1.14}
ACTIVITY_LOSS_FACTOR = {'low': 0.85, 'medium': 1.00, 'high': 1.12}

WEEKLY_MIN = 20
WEEKLY_MAX = 50
print(f'TensorFlow version: {tf.__version__}')


In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371
    p1, p2 = radians(lat1), radians(lat2)
    dp, dl = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dp / 2) ** 2 + cos(p1) * cos(p2) * sin(dl / 2) ** 2
    return r * 2 * atan2(sqrt(a), sqrt(1 - a))

df = pd.read_csv('train.csv')
df.columns = df.columns.str.strip()
for c in df.select_dtypes('object').columns:
    df[c] = df[c].astype(str).str.strip()

df.replace({'NaN': np.nan, 'conditions NaN': np.nan}, inplace=True)
df['Weatherconditions'] = df['Weatherconditions'].str.replace('conditions ', '', regex=False)
df['Time_taken_min'] = df['Time_taken(min)'].str.replace('(min) ', '', regex=False).astype(float)

df['Delivery_person_Age'] = pd.to_numeric(df['Delivery_person_Age'], errors='coerce')
df['Delivery_person_Ratings'] = pd.to_numeric(df['Delivery_person_Ratings'], errors='coerce')
df['multiple_deliveries'] = pd.to_numeric(df['multiple_deliveries'], errors='coerce').fillna(0).astype(int)
df['Festival'] = df['Festival'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)
df['Vehicle_condition'] = pd.to_numeric(df['Vehicle_condition'], errors='coerce')

df.dropna(subset=['City', 'Weatherconditions', 'Road_traffic_density', 'Type_of_vehicle'], inplace=True)
for col in ['Delivery_person_Age', 'Delivery_person_Ratings', 'Vehicle_condition']:
    df[col].fillna(df[col].median(), inplace=True)

df['distance_km'] = df.apply(
    lambda r: haversine_km(
        r['Restaurant_latitude'],
        r['Restaurant_longitude'],
        r['Delivery_location_latitude'],
        r['Delivery_location_longitude'],
    ),
    axis=1,
)

def time_bucket(t):
    try:
        h = int(str(t).split(':')[0])
        if 5 <= h < 12:
            return 'Morning'
        if 12 <= h < 17:
            return 'Afternoon'
        if 17 <= h < 22:
            return 'Evening'
        return 'Night'
    except Exception:
        return 'Afternoon'

df['time_of_day'] = df['Time_Orderd'].apply(time_bucket)
df['is_night_shift'] = (df['time_of_day'] == 'Night').astype(int)
print(f'{len(df):,} clean records ready for training')


In [ ]:
fdt = pd.read_csv('Food_Delivery_Times.csv').dropna(subset=['Courier_Experience_yrs'])
exp_mean, exp_std = fdt['Courier_Experience_yrs'].mean(), fdt['Courier_Experience_yrs'].std()

np.random.seed(42)
df['experience_yrs'] = np.clip(np.random.normal(exp_mean, exp_std, len(df)), 0, 9)
df['experience_tier'] = pd.cut(df['experience_yrs'], bins=[-0.1, 1, 3, 6, 9], labels=[0, 1, 2, 3]).astype(float)

df['daily_income_inr'] = (
    df['City'].map(CITY_INCOME).fillna(500)
    + df['multiple_deliveries'] * 80
    + df['Festival'] * 150
    + df['is_night_shift'] * 80
    - (2 - df['Vehicle_condition']) * 30
).clip(lower=200, upper=1200)

def infer_peril(row):
    weather = str(row['Weatherconditions'])
    traffic = str(row['Road_traffic_density'])
    if weather in ['Stormy', 'Sandstorms']:
        return 'weather'
    if weather == 'Fog':
        return 'aqi'
    if traffic in ['Jam', 'High']:
        return 'traffic'
    return 'platform_outage'

def infer_activity(row):
    if row['multiple_deliveries'] >= 2 or row['is_night_shift'] == 1:
        return 'high'
    if row['multiple_deliveries'] <= 0:
        return 'low'
    return 'medium'

df['peril_type'] = df.apply(infer_peril, axis=1)
df['activity_tier'] = df.apply(infer_activity, axis=1)

def compute_premium(row):
    city = row['City']
    peril = row['peril_type']
    activity = row['activity_tier']

    trigger_probability = CITY_TRIGGER_BASE.get(city, 0.14) * PERIL_TRIGGER_FACTOR.get(peril, 1.0) * ACTIVITY_TRIGGER_FACTOR.get(activity, 1.0)
    trigger_probability = float(np.clip(trigger_probability, 0.05, 0.45))

    avg_income_lost_per_day = row['daily_income_inr'] * PERIL_LOSS_FRACTION.get(peril, 0.2) * ACTIVITY_LOSS_FACTOR.get(activity, 1.0)
    days_exposed = PERIL_EXPOSURE_DAYS.get(peril, 2)

    base_premium = trigger_probability * avg_income_lost_per_day * days_exposed
    adjusted = base_premium * CITY_FACTOR.get(city, 1.0) * PERIL_FACTOR.get(peril, 1.0) * ACTIVITY_FACTOR.get(activity, 1.0)

    return float(np.round(np.clip(adjusted, WEEKLY_MIN, WEEKLY_MAX), 2))

df['premium_inr'] = df.apply(compute_premium, axis=1)
print(f"Premium range | min={df['premium_inr'].min():.0f} max={df['premium_inr'].max():.0f} mean={df['premium_inr'].mean():.1f}")


In [ ]:
def build_features(row):
    return [
        (row['Delivery_person_Age'] - 15) / 45.0,
        (row['Delivery_person_Ratings'] - 1.0) / 4.0,
        row['Vehicle_condition'] / 3.0,
        min(row['multiple_deliveries'], 3) / 3.0,
        float(row['Festival']),
        float(row['is_night_shift']),
        min(row['distance_km'], 30.0) / 30.0,
        row['experience_tier'] / 3.0,
        CITY_ENC.get(row['City'], 0.5),
        WEATHER_ENC.get(row['Weatherconditions'], 0.33),
        TRAFFIC_ENC.get(row['Road_traffic_density'], 0.33),
        VEHICLE_ENC.get(row['Type_of_vehicle'], 0.5),
        TOD_ENC.get(row['time_of_day'], 0.33),
        min(row['daily_income_inr'], 1200) / 1200.0,
    ]

X = np.array(df.apply(build_features, axis=1).tolist(), dtype=np.float32)
y = df['premium_inr'].values.astype(np.float32)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Features: {X.shape} | Train: {len(X_train):,} | Test: {len(X_test):,}')


In [ ]:
tf.random.set_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(14,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='linear'),
], name='InsureGig_PremiumModel_Phase2')

model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse', metrics=['mae'])
model.summary()

history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=80,
    batch_size=256,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    ],
    verbose=1,
)
print('Training complete')


In [ ]:
preds = model.predict(X_test).flatten()
rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
mae = float(mean_absolute_error(y_test, preds))
r2 = float(r2_score(y_test, preds))
print(f'RMSE Rs{rmse:.2f} | MAE Rs{mae:.2f} | R2 {r2:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Loss Curve')
axes[0].legend()

sample = np.random.choice(len(y_test), min(300, len(y_test)), replace=False)
axes[1].scatter(y_test[sample], preds[sample], alpha=0.4, s=10, color='#2563eb')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title('Predicted vs Actual Premium')
plt.tight_layout()
plt.show()


In [ ]:
import tensorflowjs as tfjs

os.makedirs('premium_model', exist_ok=True)
tfjs.converters.save_keras_model(model, 'premium_model')

config = {
    'feature_names': [
        'age', 'ratings', 'vehicle_condition', 'multiple_deliveries', 'festival',
        'is_night_shift', 'distance_km', 'experience_tier', 'city', 'weather',
        'traffic', 'vehicle', 'time_of_day', 'daily_income'
    ],
    'weekly_min': 20,
    'weekly_max': 50,
    'formula': 'trigger_probability * avg_income_lost_per_day * days_exposed with city/peril/activity adjustments',
    'encodings': {
        'city': CITY_ENC,
        'weather': WEATHER_ENC,
        'traffic': TRAFFIC_ENC,
        'vehicle': VEHICLE_ENC,
        'time_of_day': TOD_ENC
    },
    'city_base_income': CITY_INCOME,
    'model_info': {
        'trained_on': f'{len(df):,} real Indian delivery records',
        'rmse_inr': round(rmse, 2),
        'mae_inr': round(mae, 2),
        'r2': round(r2, 4)
    }
}

with open('premium_model/model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Exported files:')
for fname in os.listdir('premium_model'):
    print(f'  {fname}')


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('premium_model', 'zip', '.', 'premium_model')
files.download('premium_model.zip')

print('Download complete.')
print('Extract into: D:/GigInc/public/premium_model/')
print('Required files: model.json, group1-shard1of1.bin, model_config.json')
